# UBER Pickups 

## 0 - Data set
Sur Kaggle on peut avoir quelques informations complémentaires sur la structure des données.
https://www.kaggle.com/datasets/fivethirtyeight/uber-pickups-in-new-york-city

### Uber trip data from 2014

There are six files of raw data on Uber pickups in New York City from April to September 2014. The files are separated by month and each has the following columns:

    Date/Time : The date and time of the Uber pickup
    Lat : The latitude of the Uber pickup
    Lon : The longitude of the Uber pickup
    Base : The TLC base company code affiliated with the Uber pickup

These files are named:

    uber-raw-data-apr14.csv
    uber-raw-data-aug14.csv
    uber-raw-data-jul14.csv
    uber-raw-data-jun14.csv
    uber-raw-data-may14.csv
    uber-raw-data-sep14.csv

### Uber trip data from 2015

Also included is the file uber-raw-data-janjune-15.csv This file has the following columns:

    Dispatching_base_num : The TLC base company code of the base that dispatched the Uber
    Pickup_date : The date and time of the Uber pickup
    Affiliated_base_num : The TLC base company code affiliated with the Uber pickup
    locationID : The pickup location ID affiliated with the Uber pickup

The Base codes are for the following Uber bases:

- B02512 : Unter
- B02598 : Hinter
- B02617 : Weiter
- B02682 : Schmecken
- B02764 : Danach-NY
- B02765 : Grun
- B02835 : Dreist
- B02836 : Drinnen

For coarse-grained location information from these pickups, the file taxi-zone-lookup.csv shows the taxi Zone (essentially, neighborhood) and Borough for each locationID.



https://scikit-learn.org/stable/modules/generated/sklearn.cluster.HDBSCAN.html

https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html#sklearn.cluster.DBSCAN

## 1 - Exploration à partir des méta données sur le dataset

Tout d'abord on voit que
- sur 2014 on a 6 mois consécutifs et complets de avril à septembre avec des localisation et une base taxi
- sur 2015 on également 6 mois mais les données n'ont pas la géolocalisation et n'ont qu'une locID correspondant à un quartier et une base


### 1.1 - Nombre de courses / mois
On va tout d'abord quantifier les données présentes dans les datasets. Etant donné qu'il sont volumineux (4,5millions de lignes pour 2014 et plus de 14 millions pour 2015), on applique un script en Bash Shell qui effectue un comptage simple du nombre de lignes pour chaque mois.

In [2]:
# avec un traitement bash shell à base de simple commande wc et grep
# on récupère le nombre de ligne par mois -> vers fichier de sortie stats_by_month.csv
!bash tools/stats.sh


Traitement fichiers 2014...
Mois: apr
Mois: aug
Mois: jul
Mois: jun
Mois: may
Mois: sep
Traitement fichiers 2015...
Mois: 01
Mois: 02
Mois: 03
Mois: 04
Mois: 05
Mois: 06
Mois: 07
Mois: 08
Mois: 09
Mois: 10
Mois: 11
Mois: 12


In [3]:
import time
from pathlib import Path
from joblib import Parallel, delayed

import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point

from sklearn.cluster import KMeans, DBSCAN, HDBSCAN

import plotly.express as px
import plotly.graph_objects as go 
import plotly.io as pio
print(pio.renderers.default)
#pio.renderers.default = "iframe_connected"
pio.renderers.default = "vscode"
Path("exports").mkdir(exist_ok=True)
print("Renderer par défaut :", pio.renderers.default)
print("Renderers disponibles :", pio.renderers)



vscode
Renderer par défaut : vscode
Renderers disponibles : Renderers configuration
-----------------------
    Default renderer: 'vscode'
    Available renderers:
        ['plotly_mimetype', 'jupyterlab', 'nteract', 'vscode',
         'notebook', 'notebook_connected', 'kaggle', 'azure', 'colab',
         'cocalc', 'databricks', 'json', 'png', 'jpeg', 'jpg', 'svg',
         'pdf', 'browser', 'firefox', 'chrome', 'chromium', 'iframe',
         'iframe_connected', 'sphinx_gallery', 'sphinx_gallery_png']



In [4]:
df_stats_month = pd.read_csv("data/stats_by_month.csv")
#display(df_stats_month)

df_stats_month['date'] = pd.to_datetime(
    df_stats_month['year'].astype(str)
    + '-'
    + df_stats_month['month'].astype(str).str.zfill(2)
    + '-01'
)
df_stats_month['year_category'] = df_stats_month['year'].astype(str)
df_stats_month.rename(columns={'nlines':'pickups'}, inplace=True)
display(df_stats_month)


,year,month,pickups,date,year_category
0,2014,4,564516,2014-04-01,2014
1,2014,5,652435,2014-05-01,2014
2,2014,6,663844,2014-06-01,2014
3,2014,7,796121,2014-07-01,2014
4,2014,8,829275,2014-08-01,2014
5,2014,9,1028136,2014-09-01,2014
6,2015,1,1953800,2015-01-01,2015
7,2015,2,2263619,2015-02-01,2015
8,2015,3,2259772,2015-03-01,2015
9,2015,4,2280836,2015-04-01,2015


In [5]:
fig = px.bar(df_stats_month,
             x=df_stats_month['date'],
             y=df_stats_month['pickups'],
             color='year_category',
             barmode='group',
             title="Nombre de courses Uber par mois et année",
             labels={ 'nlines':'Nombre de courses', 'date':'Date', 'year_category':'Année'},
             width=1000, height=600
)
fig.update_xaxes(
    dtick="M1",  # Un tick par mois
    tickformat="%b %Y",  # Format: Jan 2014, Feb 2014, etc.
    tickangle=-45  # Angle pour une meilleure lisibilité
)

# on ajoute une régression linéaire des valeurs pour voir la tendance
df_stats_month['date_num'] = (df_stats_month['date'] - df_stats_month['date'].min()).dt.days
z = np.polyfit(df_stats_month['date_num'], df_stats_month['pickups'], deg=2)
p = np.poly1d(z)
y_trend = p(df_stats_month['date_num'])
fig.add_trace(go.Scatter(x=df_stats_month['date'], y=y_trend, mode="lines", name='Tendance', line=dict(color='red', dash='dash')))

fig.show()
fig.write_image("exports/courses_par_mois.png", scale=2)

### 1.2 - Identification et estimation des positions géographiques des bases en 2014
Les données de 2015 n'étant pas géolocalisées et uniquement affectées à des bases, on tente de voir si on peut estimer la position des bases dans les fichiers de 2014.

In [6]:
# avec un script python on calcul la position moyenne+std des bases
!python tools/compute_base_locations.py

Coordonnées moyennes enregistrées dans data/computed_base_locations_2014.csv


In [7]:
df_base_2014 = pd.read_csv('data/computed_base_locations_2014.csv')

df_base_2014.head()

,Base,Lat_Moy,Lon_Moy,Lat_Std,Lon_Std
0,B02512,40.744196,-73.976005,0.040875,0.071978
1,B02598,40.738571,-73.971787,0.041160,0.059137
2,B02617,40.739244,-73.972048,0.039915,0.057733
3,B02682,40.739240,-73.972369,0.040346,0.057715
4,B02764,40.739072,-73.969949,0.042707,0.056032


In [8]:
fig = go.Figure()

# fond de carte avec position moy des bases
fig = px.scatter_map(
    df_base_2014,
    lat="Lat_Moy",
    lon="Lon_Moy",
    hover_name="Base",
    zoom=10,
    center={"lat": 40.7128, "lon": -74.0060},  # New York
    title="Positions moyennes des bases TLC 2014 avec incertitude (±1σ)",
)

# création des bounding box autour des stations
for _, row in df_base_2014.iterrows():
    lon, lat = row["Lon_Moy"], row["Lat_Moy"]
    std_lon, std_lat = row["Lon_Std"], row["Lat_Std"]

    # coordonnées de la bounding box avec l'écart-type
    factor = 1
    lons = [lon - factor*std_lon, lon - factor*std_lon, lon + factor*std_lon, lon + factor*std_lon, lon - factor*std_lon]
    lats = [lat - factor*std_lat, lat + factor*std_lat, lat + factor*std_lat, lat - factor*std_lat, lat - factor*std_lat]

    fig.add_trace(go.Scattermap(
        lon=lons,
        lat=lats,
        mode="lines",
        fill="none",
        fillcolor="rgba(0, 100, 255, 0.2)",
        line=dict(color="red", width=2),
        hoverinfo="none",
        showlegend=False
    ))

map_style="carto-positron"
#map_style="open-street-map"
fig.update_layout(
    map=dict(center=dict(lat=40.7128, lon=-74.0060), style=map_style, zoom=10),
    margin=dict(r=0, t=40, l=0, b=0)
)

fig.show()
fig.write_html("exports/bases_tlc_2014.html")


Visiblement impossible de rapprocher une course d'une base : trop vaste => on écarte donc le jeu de données de 2015.

### 1.3 - Chargement de toutes les données 2014


In [9]:
def load_csv_uber(file:Path)->pd.DataFrame:
    ''' Méthode pour charger un fichier Uber 2014'''
    print(f"Read {file}...")
    df = pd.read_csv(file, parse_dates=[0], date_format={"Date/Time":"%m/%d/%Y %H:%M:%S"})
    df.rename(columns={ k: c.replace('/','').lower() for k,c in zip(df.columns, df.columns)}, inplace=True)
    return df

input_dir = Path("data/uber-trip-data/2014")
file_list = list(input_dir.glob("*.csv"))

start = time.time()
executor = Parallel(n_jobs=6, backend='loky')
df_list = executor(delayed(load_csv_uber)(f) for f in file_list)
print(f"Temps total: {time.time() - start:.2f}s")

df = pd.concat(df_list, ignore_index=True)


Read data/uber-trip-data/2014/uber-raw-data-sep14.csv...
Read data/uber-trip-data/2014/uber-raw-data-jul14.csv...
Read data/uber-trip-data/2014/uber-raw-data-aug14.csv...
Read data/uber-trip-data/2014/uber-raw-data-jun14.csv...
Read data/uber-trip-data/2014/uber-raw-data-apr14.csv...
Read data/uber-trip-data/2014/uber-raw-data-may14.csv...
Temps total: 3.25s


In [10]:
# définition des mappings
dow_map = {
    1: 'Lundi', 2: 'Mardi', 3: 'Mercredi', 4: 'Jeudi',
    5: 'Vendredi', 6: 'Samedi', 7: 'Dimanche'
}
month_map = {
    '01': 'Janvier', '02': 'Février', '03': 'Mars', '04': 'Avril', '05': 'Mai', '06': 'Juin',
    '07': 'Juillet', '08': 'Août', '09':'Septembre', '10':'Octobre', '11':'Novembre', '12':'Décembre'
}

In [11]:
# ajout de colonnes pour avoir le mois, le jour, num jour de semaine, nom jour de semaine
df['month'] = df['datetime'].dt.strftime("%m")
df['day'] = df['datetime'].dt.strftime("%d")
df['month_str'] = df['datetime'].dt.strftime("%b")
df['hour'] = df['datetime'].dt.strftime("%H")
df['dow'] = df['datetime'].dt.strftime("%u").astype(int)
df['dow_name'] = df['dow'].map(dow_map)
display(df.head())
df.info()

,datetime,lat,lon,base,month,day,month_str,hour,dow,dow_name
0,2014-09-01 00:01:00,40.2201,-74.0021,B02512,09,01,Sep,00,1,Lundi
1,2014-09-01 00:01:00,40.7500,-74.0027,B02512,09,01,Sep,00,1,Lundi
2,2014-09-01 00:03:00,40.7559,-73.9864,B02512,09,01,Sep,00,1,Lundi
3,2014-09-01 00:06:00,40.7450,-73.9889,B02512,09,01,Sep,00,1,Lundi
4,2014-09-01 00:11:00,40.8145,-73.9444,B02512,09,01,Sep,00,1,Lundi


<class 'pandas.DataFrame'>
RangeIndex: 4534327 entries, 0 to 4534326
Data columns (total 10 columns):
 #   Column     Dtype         
---  ------     -----         
 0   datetime   datetime64[us]
 1   lat        float64       
 2   lon        float64       
 3   base       str           
 4   month      str           
 5   day        str           
 6   month_str  str           
 7   hour       str           
 8   dow        int64         
 9   dow_name   str           
dtypes: datetime64[us](1), float64(2), int64(1), str(6)
memory usage: 345.9 MB


In [12]:
df.isna().any()

datetime     False
lat          False
lon          False
base         False
month        False
day          False
month_str    False
hour         False
dow          False
dow_name     False
dtype: bool

In [13]:
df.describe(include="all")

,datetime,lat,lon,base,month,day,month_str,hour,dow,dow_name
count,4534327,4.534327e+06,4.534327e+06,4534327,4534327,4534327,4534327,4534327,4.534327e+06,4534327
unique,NaN,NaN,NaN,5,6,31,6,24,NaN,7
top,NaN,NaN,NaN,B02617,09,30,Sep,17,NaN,Jeudi
freq,NaN,NaN,NaN,1458853,1028136,167160,1028136,336190,NaN,755145
mean,2014-07-11 18:50:50.578152,4.073926e+01,-7.397302e+01,NaN,NaN,NaN,NaN,NaN,3.968115e+00,NaN
min,2014-04-01 00:00:00,3.965690e+01,-7.492900e+01,NaN,NaN,NaN,NaN,NaN,1.000000e+00,NaN
25%,2014-05-28 15:18:00,4.072110e+01,-7.399650e+01,NaN,NaN,NaN,NaN,NaN,2.000000e+00,NaN
50%,2014-07-17 14:45:00,4.074220e+01,-7.398340e+01,NaN,NaN,NaN,NaN,NaN,4.000000e+00,NaN
75%,2014-08-27 21:55:00,4.076100e+01,-7.396530e+01,NaN,NaN,NaN,NaN,NaN,6.000000e+00,NaN
max,2014-09-30 22:59:00,4.211660e+01,-7.206660e+01,NaN,NaN,NaN,NaN,NaN,7.000000e+00,NaN


### 1.4 - Analyse du trafic / jour de semaine et /heure

In [14]:
df_by_dow = df.groupby(['dow','dow_name']).count()['datetime'].reset_index()
display(df_by_dow)

,dow,dow_name,datetime
0,1,Lundi,541472
1,2,Mardi,663789
2,3,Mercredi,696488
3,4,Jeudi,755145
4,5,Vendredi,741139
5,6,Samedi,646114
6,7,Dimanche,490180


In [15]:
fig = px.line(
    data_frame=df_by_dow,
    x='dow', y='datetime',
    labels={'dow':'Jour de semaine','datetime':'Nb pickups'})
fig.update_xaxes(
    tickmode='array',
    tickvals= list(dow_map.keys()),
    ticktext= list(dow_map.values()),
)
fig.show()
fig.write_image("exports/pickups_par_dow.png", scale=2)

- On voit sur le graphe que les jours les plus chargés sont le jeudi et vendredi.
- Le jeudi est le jour le plus chargé avec 750.000 courses (~17%)
- du mardi au samedi on a 3/4 des courses

In [16]:
df_by_hour = df.groupby('hour').count()['datetime'].reset_index()
display(df_by_hour)

,hour,datetime
0,00,103836
1,01,67227
2,02,45865
3,03,48287
4,04,55230
5,05,83939
6,06,143213
7,07,193094
8,08,190504
9,09,159967


In [17]:
fig = px.line(data_frame=df_by_hour, x='hour', y='datetime', labels=dict(hour='Heure',datetime='Nb pickups'))
fig.show()
fig.write_image("exports/pickups_par_heure.png", scale=2)

On distingue nettement:
- **une Heure de Pointe du Matin (HPM) de 6 à 9h**
- **une Heure de Pointe du Soir (HPS) de 16 à 19h**
En réalité l'HPS est plus diffuse et on pourrait même considérer qu'elle s'étend de 15 à 21h

In [18]:
# HPM de 6h à 9h
# HPS plus diffuse de 16h 19h
#df_by_hour['cat'] = df_by_hour['hour']
df_by_hour['cat'] = df_by_hour['hour'].apply(lambda x: "HPM" if (x>="06") & (x<="09") else "HPS" if (x>="16") & (x<="19") else "")
display(df_by_hour)
#fig = px.histogram(data_frame=df_by_hour, x='hour', y='datetime', cumulative=True) #, color='cat', barmode="relative")
#fig.show()

,hour,datetime,cat
0,00,103836,
1,01,67227,
2,02,45865,
3,03,48287,
4,04,55230,
5,05,83939,
6,06,143213,HPM
7,07,193094,HPM
8,08,190504,HPM
9,09,159967,HPM


In [19]:
#df_by_hour_dow = df.groupby(['hour','dow_name'])['datetime'].count().reset_index()
df_by_hour_dow = df.groupby(['hour','dow'])['datetime'].agg(pickups='count').reset_index()
display(df_by_hour_dow)

fig = px.density_heatmap(df_by_hour_dow, 
                         x='dow', 
                         y='hour', 
                         z='pickups',
                         labels={'dow': 'Jour semaine', 'hour': 'Heure', 'pickups': 'Nombre de pickups'},
                         color_continuous_scale='YlOrRd',
                         title='Heatmap des pickups jour de semaine x heure')

# Remplacer les numéros par les noms de jours
fig.update_xaxes(
    tickmode='array',
    tickvals= list(dow_map.keys()),
    ticktext= list(dow_map.values()),
)

fig.show()
fig.write_image("exports/heatmap_pickups.png", scale=2)

,hour,dow,pickups
0,00,1,6436
1,00,2,6237
2,00,3,7644
3,00,4,9293
4,00,5,13716
...,...,...,...
163,23,3,18146
164,23,4,27764
165,23,5,41260
166,23,6,43174


- On voit que le week-end + lundi sont calmes
- le reste est concentré sur la semaine, et surtout jeudi et vendredi
- On voit que l'heure de pointe du matin (HPM) et l'heure de pointe du soir (HPS) restent marquées aux mêmes heures sur la semaine, sauf pour le week-end pour laquelle l'HPM disparait.

**On pourrait alors découper la journée en 4 partie :**
- **HPM matin de 6 à 9h**
- **HC jour de 9 à 16h**
- **HPS soir de 16h à 19h**
- **HC nuit de 19h à 6h**


## 2 - Modélisation baseline

### 2.1 - Réduction du dataset pour un premier modèle

Etant donné le dataset trop volumineux, et le fait que les déplacements peuvent etre radicalement différents entre le matin et le soir, on va regarder uniquement :
- pour le mois de **septembre** (le plus chargé)
- le **jeudi** qui est la journée type la plus chargée
- et sur **l'HPS de 16h à 19h**.


In [20]:
#MAP_STYLE="open-street-map"
MAP_STYLE="carto-positron"
NY_CENTER = {"lat": 40.7128, "lon": -74.0060}
SAMPLE_N_VIZ = 1000  # None = tout le sous-ensemble / réduire pour avoir moins de- points

# définition des masks de filtrage
mask_thursday = df['dow']==4
mask_sept = df['month']=='09'
mask_hps = (df['hour']>="16") & (df['hour']<="19")
df_thurs_hps = df[ mask_thursday & mask_sept & mask_hps ].copy()
print(f"df_thurs_hps initial : {len(df_thurs_hps)}")

# échantillonage
if SAMPLE_N_VIZ is not None and SAMPLE_N_VIZ < len(df_thurs_hps):
    df_thurs_hps = df_thurs_hps.sample(n=SAMPLE_N_VIZ, random_state=42).copy()
print(f"df_thurs_hps : {len(df_thurs_hps)} pts")
df_thurs_hps

df_thurs_hps initial : 44194
df_thurs_hps : 1000 pts


,datetime,lat,lon,base,month,day,month_str,hour,dow,dow_name
986479,2014-09-25 18:16:00,40.7021,-74.0110,B02764,09,25,Sep,18,4,Jeudi
235783,2014-09-25 18:36:00,40.7716,-73.9894,B02598,09,25,Sep,18,4,Jeudi
415072,2014-09-11 19:41:00,40.7686,-73.8627,B02617,09,11,Sep,19,4,Jeudi
59616,2014-09-04 17:36:00,40.7545,-73.9948,B02598,09,04,Sep,17,4,Jeudi
818317,2014-09-25 19:56:00,40.6936,-73.9642,B02682,09,25,Sep,19,4,Jeudi
...,...,...,...,...,...,...,...,...,...,...
415083,2014-09-11 19:42:00,40.6467,-73.7899,B02617,09,11,Sep,19,4,Jeudi
414157,2014-09-11 18:51:00,40.7648,-73.9816,B02617,09,11,Sep,18,4,Jeudi
939761,2014-09-18 19:04:00,40.7564,-73.9664,B02764,09,18,Sep,19,4,Jeudi
317121,2014-09-04 18:11:00,40.7450,-73.9801,B02617,09,04,Sep,18,4,Jeudi


In [21]:
fig = px.scatter_map(
    df_thurs_hps,
    lat="lat",
    lon="lon",
    hover_name="base",
    zoom=10,
    center=NY_CENTER,
    title="Courses pour les jeudis de septembre de 16h à 19h",
)

fig.update_layout(
    map=dict(center=NY_CENTER, style=MAP_STYLE, zoom=10),
    margin=dict(r=0, t=40, l=0, b=0)
)

fig.show()
fig.write_html("exports/scatter_jeudis_sept.html")

### 2.2 - Utilisation de KMeans

On va tenter une clusteristation par KMeans

In [22]:
# Transformer WGS84 (EPSG:4326) vers une projection métrique
# Pour la France métropolitaine : Lambert 93 (EPSG:2154)
# Pour NYC/USA Est : UTM zone appropriée (ex: EPSG:32618)
#conv_WGS84_to_UTM = Transformer.from_crs("EPSG:4326","EPSG:32618", always_xy=True)
#conv_UTM_to_WGS84 = Transformer.from_crs("EPSG:32618", "EPSG:4326", always_xy=False)
crs_WGS84 = 'EPSG:4326'
crs_UTM18N = 'EPSG:32618'

geometry = [ Point(lon, lat) for lon, lat in zip(df_thurs_hps['lon'], df_thurs_hps['lat'])]
gfd_thurs_hps = gpd.GeoDataFrame(df_thurs_hps, geometry=geometry, crs=crs_WGS84)
gfd_thurs_hps_UTM = gfd_thurs_hps.to_crs(crs_UTM18N)

# ajout des coord converties X,Y dans le dataset
#df_thurs_hps['x'], df_thurs_hps['y'] = coord_converter.transform(df['lon'].values, df_thurs_hps['lat'].values)
df_thurs_hps.loc[:, 'x_m'] = gfd_thurs_hps_UTM.geometry.x.values
df_thurs_hps.loc[:, 'y_m'] = gfd_thurs_hps_UTM.geometry.y.values

# préparation du jeu d'entrainement
X = df_thurs_hps[['x_m','y_m']].values

# calcul des KMeans et récupération des centroides de cluster
kmeans = KMeans(n_clusters=20, random_state=0, max_iter=1000)

df_thurs_hps.loc[:, 'cluster'] = kmeans.fit_predict(X)
df_thurs_hps.loc[:, 'cluster_str'] = df_thurs_hps['cluster'].astype(str)

# Récupérer les centres
centers_xy = kmeans.cluster_centers_
gdf_centroids = gpd.GeoDataFrame(geometry=[Point(x,y) for x,y in centers_xy], crs=crs_UTM18N)
gdf_centroids_WGS84 = gdf_centroids.to_crs(crs_WGS84)

centers_long = gdf_centroids_WGS84.geometry.x.values
centers_lat = gdf_centroids_WGS84.geometry.y.values



In [23]:
# Taille de chaque cluster → colonne de couleur pour les points
cluster_sizes = df_thurs_hps['cluster'].value_counts().sort_index()
df_thurs_hps.loc[:, 'cluster_size'] = df_thurs_hps['cluster'].map(cluster_sizes)

# Carte avec les points colorés par importance du cluster
fig = px.scatter_map(df_thurs_hps,
                        lat='lat',
                        lon='lon',
                        color='cluster_size',
                        color_continuous_scale='YlOrRd',
                        zoom=10,
                        height=700,
                        width=1000,
                        opacity=0.6,
                        labels={'cluster_size': 'Pickups'})

# Ajouter les centres des clusters (noir fixe)
sizes = [int(cluster_sizes.get(i, 0)) for i in range(len(centers_lat))]
fig.add_trace(go.Scattermap(
    lat=centers_lat,
    lon=centers_long,
    mode='markers+text',
    marker=dict(size=18, color='black', symbol='circle'),
    name='Centres des clusters',
    text=[f'Centre {i}' for i in range(len(centers_lat))],
    textfont=dict(size=14, color='white', family='Arial Black'),
    hovertext=[f'Centroïde {i}<br>Pickups : {s}<br>Lat: {lat:.4f}<br>Lon: {lon:.4f}'
               for i, (s, lat, lon) in enumerate(zip(sizes, centers_lat, centers_long))],
    showlegend=True
))

fig.update_layout(map_style=map_style)
fig.show()
fig.write_html("exports/kmeans_jeudis_sept_hps.html")


In [24]:
cluster_sizes

cluster
0     261
1      42
2      98
3       1
4     200
5      12
6       1
7       7
8       1
9       9
10      8
11     35
12      7
13     46
14      1
15     95
16    165
17      3
18      1
19      7
Name: count, dtype: int64

In [25]:
# Pour faire une représentation des centroides de façon proportionnelle à leur importance
min_size = 20 # taille min du centroide
max_size = 100 # taille max du centroide
sizes = min_size + (cluster_sizes.values - cluster_sizes.min()) / (cluster_sizes.max() - cluster_sizes.min()) * (max_size - min_size)
print(sizes)

centroid_colors = cluster_sizes.values.tolist()

[100.          32.61538462  49.84615385  20.          81.23076923
  23.38461538  20.          21.84615385  20.          22.46153846
  22.15384615  30.46153846  21.84615385  33.84615385  20.
  48.92307692  70.46153846  20.61538462  20.          21.84615385]


In [26]:
# Carte avec centroides de façon proportionnelle à leur importance
fig = go.Figure()

fig.add_trace(go.Scattermap(
    lat=centers_lat,
    lon=centers_long,
    mode='markers+text',
    marker=dict(size=sizes, color=centroid_colors, colorscale='YlOrRd', showscale=True, colorbar=dict(title='Pickups', thickness=12, len=0.5), symbol='circle', opacity=0.9, sizemode='diameter'),
    name='Centres des clusters',
    text=[f'Centre {i}' for i in range(len(centers_lat))],
    textfont=dict(size=14, color='white', family='Arial Black'),
    hovertext=[f'Centroïde {i}<br>Lat: {lat:.4f}<br>Lon: {lon:.4f}' 
               for i, (lat, lon) in enumerate(zip(centers_lat, centers_long))],
    showlegend=True
))
fig.update_layout(
    map=dict(
        style=map_style,
        center=dict(
            lat=centers_lat.mean(),
            lon=centers_long.mean()
        ),
        zoom=10
    ),
    height=700,
    width=1000,
    title='Centroïdes des clusters (taille = nombre de pickups)'
)
fig.show()
fig.write_html("exports/kmeans_centroids_jeudis_sept.html")

In [27]:
#
# Création des méthodes pour pouvoir généraliser
#

def extract_period_df(df:pd.DataFrame, dows:list, months:list, from_hour:str, to_hour:str, sample_n:int=None) -> pd.DataFrame:
    """Extraction d'un sous-ensemble du DataFrame selon les jours, mois et plage horaire + échantillonage
    Args:
        df        : DataFrame complet des courses Uber (colonnes 'dow', 'month', 'hour').
        dows      : Liste des jours de la semaine (0=lundi … 6=dimanche).
        months    : Liste des mois sous forme de chaînes à 2 chiffres ('01'–'12').
        from_hour : Heure de début incluse (format '00'–'23').
        to_hour   : Heure de fin incluse (format '00'–'23').
        sample_n  : Taille max de l'échantillon (None = tout conserver).

    Returns:
        Une copie du DataFrame filtré selon, jours, mois et plage horaire et éventuellement sous-échantillonné.
    """
    mask_dow = df['dow'].isin(dows)
    mask_month = df['month'].isin(months)

    if from_hour <= to_hour:
        mask_hour = (df['hour']>=from_hour) & (df['hour']<=to_hour)
    else:  # plage franchissant minuit : ex. '19'→'06'
        mask_hour = (df['hour']>=from_hour) | (df['hour']<=to_hour)

    df_reduced = df[ mask_dow & mask_month & mask_hour ].copy()

    if sample_n is not None and sample_n < len(df_reduced):
        df_reduced = df_reduced.sample(n=sample_n, random_state=42).copy()
    print(f"Extract: dow={dows} months={months} {from_hour}h–{to_hour}h → {len(df_reduced)} pts")
    
    return df_reduced

def convert_and_get_xy(df:pd.DataFrame) -> np.array:
    """Convertit les coordonnées GPS (WGS84) en mètres (UTM 18N) et les ajoute au DataFrame.
    Ajoute les colonnes 'x_m' et 'y_m' X,Y en mètres dans df (in place).

    Args:
        df : DataFrame avec colonnes 'lat' et 'lon' (WGS84).

    Returns:
        Tableau NumPy (n, 2) des coordonnées métriques (pour KMeans de scikit-learn)
    """
    geometry = [ Point(lon, lat) for lon, lat in zip(df['lon'], df['lat'])]
    gfd = gpd.GeoDataFrame(df, geometry=geometry, crs=crs_WGS84)
    gfd_UTM = gfd.to_crs(crs_UTM18N)
    df.loc[:, 'x_m'] = gfd_UTM.geometry.x.values
    df.loc[:, 'y_m'] = gfd_UTM.geometry.y.values
    X = df[['x_m','y_m']].values
    return X

def compute_cluster_and_centroids(df:pd.DataFrame, n_cluster:int) -> tuple:
    """Applique KMeans sur les coordonnées métriques et retourne les centroïdes en WGS84.

    Ajoute les colonnes 'cluster' (int) et 'cluster_str' (str) dans df (en place).
    Requiert que 'x_m' et 'y_m' existent déjà (appeler convert_and_get_xy avant).

    Args:
        df        : DataFrame avec colonnes 'x_m' et 'y_m'.
        n_cluster : Nombre de clusters KMeans.

    Returns:
        (centers_long, centers_lat) : Coordonnées GPS des centroïdes (tableaux NumPy).
    """
    X = df[['x_m','y_m']].values
    kmeans = KMeans(n_clusters=n_cluster, random_state=0, max_iter=1000)
    df.loc[:, 'cluster'] = kmeans.fit_predict(X)
    df.loc[:, 'cluster_str'] = df['cluster'].astype(str)
    centers_xy = kmeans.cluster_centers_
    gdf_centroids = gpd.GeoDataFrame(geometry=[Point(x,y) for x,y in centers_xy], crs=crs_UTM18N)
    gdf_centroids_WGS84 = gdf_centroids.to_crs(crs_WGS84)
    centers_long = gdf_centroids_WGS84.geometry.x.values
    centers_lat = gdf_centroids_WGS84.geometry.y.values
    return centers_long, centers_lat

def get_map_pickups(df:pd.DataFrame, dows, months, from_hour, to_hour) -> go.Figure:
    """Carte Plotly des points de prise en charge (sans clustering).
    Args:
        df        : DataFrame avec colonnes 'lat', 'lon', 'base'.
        dows      : Jours de la semaine (utilisés uniquement pour le titre).
        months    : Mois (utilisés uniquement pour le titre).
        from_hour : Heure de début (utilisée uniquement pour le titre).
        to_hour   : Heure de fin (utilisée uniquement pour le titre).

    Returns:
        Figure Plotly interactive (scatter map).
    """
    dows_name = "+".join([ dow_map[x] for x in dows ])
    months_name = "+".join([ month_map[x] for x in months])
    title = f"Courses pour les {dows_name}"
    title += f" de {months_name}"
    title += f" de {from_hour}h à {to_hour}h"
    fig = px.scatter_map(
        df,
        lat="lat",
        lon="lon",
        hover_name="base",
        title=title,
    )
    fig.update_layout(
        map=dict(style=MAP_STYLE, center=NY_CENTER, zoom=10),
        height=700,
        width=1000,
        margin=dict(r=0, t=40, l=0, b=0)
    )
    return fig

def get_map_clusters(df:pd.DataFrame, centers_long:np.array, centers_lat:np.array, title:str="") -> go.Figure:
    """Carte Plotly avec :
    - points colorés en gradient selon l'importance de leur cluster (nombre de pickups)
    - centroïdes colorés et dimensionnés en proportion de leur cluster

    Requiert que df contienne les colonnes 'lat', 'lon', 'cluster'.

    Args:
        df           : DataFrame avec colonnes 'lat', 'lon', 'cluster'.
        centers_long : Longitudes des centroïdes (ordre correspondant aux labels 0..n).
        centers_lat  : Latitudes des centroïdes.

    Returns:
        Figure Plotly interactive (scatter map + marqueurs centroïdes).
    """
    # Taille de chaque cluster → couleur des points et des centroïdes
    cluster_sizes = df['cluster'].value_counts().sort_index()
    df = df.copy()
    df['cluster_size'] = df['cluster'].map(cluster_sizes)

    # Points colorés par importance du cluster
    fig = px.scatter_map(df,
                        lat='lat',
                        lon='lon',
                        color='cluster_size',
                        color_continuous_scale='YlOrRd',
                        opacity=0.6,
                        labels={'cluster_size': 'Pickups'}
    )

    # Centroïdes : taille et couleur proportionnelles à leur cluster
    n = len(centers_lat)
    c_sizes = [int(cluster_sizes.get(i, 0)) for i in range(n)]
    min_s, max_s = min(c_sizes), max(c_sizes)
    range_s = max_s - min_s or 1
    marker_sizes = [15 + (s - min_s) / range_s * 25 for s in c_sizes]  # 15px … 40px

    fig.add_trace(go.Scattermap(
        lat=centers_lat,
        lon=centers_long,
        mode='markers+text',
        marker=dict(
            size=marker_sizes,
            color=c_sizes,
            colorscale='YlOrRd',
            showscale=False,
            symbol='circle',
            opacity=0.9,
            sizemode='diameter',
        ),
        name='Centres des clusters',
        text=[f'Centre {i}' for i in range(n)],
        textfont=dict(size=13, color='black', family='Arial Black'),
        hovertext=[f'Centroïde {i}<br>Pickups : {s}<br>Lat: {lat:.4f}<br>Lon: {lon:.4f}'
                for i, (s, lat, lon) in enumerate(zip(c_sizes, centers_lat, centers_long))],
        showlegend=True
    ))

    fig.update_layout(
        map=dict(style=MAP_STYLE, center=NY_CENTER, zoom=10),
        height=700,
        width=1000,
        title=title,
    )
    return fig


In [28]:
clusters_defs = [
    {
        'name': 'HPM matin jeudi septembre',
        'params': {
            'data': dict(dows=[4], months=['09'], from_hour='06', to_hour='09', sample_n=SAMPLE_N_VIZ),
            'n_cluster': 20
        }
    },
    {
        'name': 'HC journée jeudi septembre',
        'params': {
            'data':  dict(dows=[4], months=['09'], from_hour='10', to_hour='15', sample_n=SAMPLE_N_VIZ),
            'n_cluster': 20
        }
    },
    {
        'name': 'HPS soir jeudi septembre',
        'params': {
            'data':  dict(dows=[4], months=['09'], from_hour='16', to_hour='19', sample_n=SAMPLE_N_VIZ),
            'n_cluster': 20
        }
    },
    {
        'name': 'HC nuit jeudi septembre',
        'params': {
            'data':  dict(dows=[4], months=['09'], from_hour='19', to_hour='06', sample_n=SAMPLE_N_VIZ),
            'n_cluster': 20
        }
    }
]


In [29]:
for c in clusters_defs:
    print(f"---------- {c['name']} --------------")
    df_period = extract_period_df(df, **c['params']['data'])
    #fig = get_map_pickups(df_period, **c['params']['data'])
    #fig.show()
    X = convert_and_get_xy(df_period)
    #display(X)
    cntrs_long, cntrs_lat = compute_cluster_and_centroids(df_period, c['params']['n_cluster'])
    fig = get_map_clusters(df_period, cntrs_long, cntrs_lat, title=c['name'])
    fig.show()
    fname = c['name'].replace(' ', '_')
    fig.write_html(f"exports/kmeans_{fname}.html")


---------- HPM matin jeudi septembre --------------
Extract: dow=[4] months=['09'] 06h–09h → 1000 pts


---------- HC journée jeudi septembre --------------
Extract: dow=[4] months=['09'] 10h–15h → 1000 pts


---------- HPS soir jeudi septembre --------------
Extract: dow=[4] months=['09'] 16h–19h → 1000 pts


---------- HC nuit jeudi septembre --------------
Extract: dow=[4] months=['09'] 19h–06h → 1000 pts


### 2.3 - Comparaison avec DBSCAN

On teste sur le sous-ensemble jeudi septembre HPS 16h–19h  
DBSCAN ne demande pas de fixer k à l'avance : il détecte les zones denses automatiquement et classe les points isolés comme étant du **bruit** (label −1).  

Les paramètres clé sont :
- `eps` rayon de voisinage en mètres : on joue sur ce paramètre pour obtenir des hot-zones cohérentes à l'échelle d'un quartier
- `min_samples` : indique la densité minimale pour un cluster
- `metric` : la norme d'évaluation des distances qui peut être `euclidian` par défaut, mais ici on utilisera `manhattan` (tiens tiens?) qui utilise |Δx|+|Δy|, ce qui correspond mieux au distances parcourues en voiture


In [30]:
DBSCAN_SAMPLE_N = 2000  # None = tout ; réduire si un seul gros cluster
EPS_M = 100             # rayon de voisinage Manhattan en mètres (~1 pâté de maisons)
MIN_SAMPLES = 10        # densité minimale pour former un cluster
METRIC = 'manhattan'    # |Δx|+|Δy| — adapté à la grille de rues NYC


In [31]:
for c in clusters_defs:
    print(f"---------- {c['name']} --------------")
    data_params = {k: v for k, v in c['params']['data'].items() if k != 'sample_n'}
    data_params['sample_n'] = DBSCAN_SAMPLE_N
    df_period = extract_period_df(df, **data_params)
    X = convert_and_get_xy(df_period)

    dbscan = DBSCAN(eps=EPS_M, min_samples=MIN_SAMPLES, metric=METRIC, n_jobs=-1)
    labels = dbscan.fit_predict(X)

    df_period.loc[:, 'cluster'] = labels
    df_period.loc[:, 'cluster_str'] = labels.astype(str)

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = (labels == -1).sum()
    print(f"eps={EPS_M}m  min_samples={MIN_SAMPLES}  → {n_clusters} clusters  |  {n_noise} pts bruit ({n_noise/len(labels)*100:.1f}%)")

    df_clean = df_period[df_period['cluster'] >= 0].copy()
    if df_clean.empty:
        print("  ⚠ aucun cluster — ajuster EPS_M ou MIN_SAMPLES")
        continue

    centers = df_clean.groupby('cluster')[['lat', 'lon']].mean().sort_index()
    fig = get_map_clusters(df_clean, centers['lon'].values, centers['lat'].values,
                           title=f"DBSCAN — {c['name']}  (eps={EPS_M}m → {n_clusters} hot-zones)")
    fig.show()
    fname = c['name'].replace(' ', '_')
    fig.write_html(f"exports/dbscan_{fname}.html")


---------- HPM matin jeudi septembre --------------
Extract: dow=[4] months=['09'] 06h–09h → 2000 pts
eps=100m  min_samples=10  → 1 clusters  |  1988 pts bruit (99.4%)


---------- HC journée jeudi septembre --------------
Extract: dow=[4] months=['09'] 10h–15h → 2000 pts
eps=100m  min_samples=10  → 6 clusters  |  1912 pts bruit (95.6%)


---------- HPS soir jeudi septembre --------------
Extract: dow=[4] months=['09'] 16h–19h → 2000 pts
eps=100m  min_samples=10  → 12 clusters  |  1824 pts bruit (91.2%)


---------- HC nuit jeudi septembre --------------
Extract: dow=[4] months=['09'] 19h–06h → 2000 pts
eps=100m  min_samples=10  → 7 clusters  |  1891 pts bruit (94.5%)


### 2.4 - HDBSCAN

**HDBSCAN** (Hierarchical DBSCAN) améliore DBSCAN sur un point clé : il gère les clusters de **densité variable**.
DBSCAN utilise un rayon fixe `eps` — si certaines zones sont denses et d'autres dispersées, un seul `eps` ne convient pas à tous.
HDBSCAN construit d'abord une hiérarchie de clusters à toutes les échelles, puis sélectionne automatiquement les clusters les plus **stables**.

Paramètres principaux :
- `min_cluster_size` : taille minimale d'un cluster (paramètre principal)
- `min_samples` *(optionnel, défaut = min_cluster_size)* : densité locale requise pour être un point «core» — augmenter rend le modèle plus conservateur (plus de bruit)
- `cluster_selection_epsilon` *(optionnel)* : fusionne les clusters distants de moins de cette valeur — utile pour contrôler la granularité finale
- `metric` : même raisonnement que DBSCAN — `manhattan` pour la grille NYC

Avantages vs DBSCAN : pas besoin de calibrer `eps`, détecte naturellement des clusters de tailles et densités différentes.


In [32]:
from sklearn.cluster import HDBSCAN

HDBSCAN_SAMPLE_N = 2000
MIN_CLUSTER_SIZE = 50    # taille minimale d'un cluster
MIN_SAMPLES = 5          # densité locale requise pour être un point core
CLUSTER_EPS = 0          # 0 = pas de fusion forcée ; en mètres si > 0
SELECTION_METHOD = 'leaf'  # 'leaf' = clusters granulaires ; 'eom' = clusters stables (défaut, trop larges)

for c in clusters_defs:
    print(f"---------- {c['name']} --------------")
    data_params = {k: v for k, v in c['params']['data'].items() if k != 'sample_n'}
    data_params['sample_n'] = HDBSCAN_SAMPLE_N
    df_period = extract_period_df(df, **data_params)
    X = convert_and_get_xy(df_period)

    hdb = HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE,
                  min_samples=MIN_SAMPLES,
                  cluster_selection_epsilon=CLUSTER_EPS,
                  cluster_selection_method=SELECTION_METHOD,
                  metric=METRIC,
                  n_jobs=-1)
    labels = hdb.fit_predict(X)

    df_period.loc[:, 'cluster'] = labels
    df_period.loc[:, 'cluster_str'] = labels.astype(str)

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = (labels == -1).sum()
    print(f"min_cluster_size={MIN_CLUSTER_SIZE}  min_samples={MIN_SAMPLES}  method={SELECTION_METHOD}  → {n_clusters} clusters  |  {n_noise} pts bruit ({n_noise/len(labels)*100:.1f}%)")

    df_clean = df_period[df_period['cluster'] >= 0].copy()
    if df_clean.empty:
        print("  ⚠ aucun cluster — ajuster MIN_CLUSTER_SIZE")
        continue

    centers = df_clean.groupby('cluster')[['lat', 'lon']].mean().sort_index()
    fig = get_map_clusters(df_clean, centers['lon'].values, centers['lat'].values,
                           title=f"HDBSCAN — {c['name']}  ({SELECTION_METHOD}, min_cluster_size={MIN_CLUSTER_SIZE} → {n_clusters} hot-zones)")
    fig.show()
    fname = c['name'].replace(' ', '_')
    fig.write_html(f"exports/hdbscan_{fname}.html")


---------- HPM matin jeudi septembre --------------
Extract: dow=[4] months=['09'] 06h–09h → 2000 pts
min_cluster_size=50  min_samples=5  method=leaf  → 8 clusters  |  792 pts bruit (39.6%)


/home/gviel/app/miniconda3/envs/uber/lib/python3.12/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


---------- HC journée jeudi septembre --------------
Extract: dow=[4] months=['09'] 10h–15h → 2000 pts
min_cluster_size=50  min_samples=5  method=leaf  → 10 clusters  |  1025 pts bruit (51.2%)


/home/gviel/app/miniconda3/envs/uber/lib/python3.12/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


---------- HPS soir jeudi septembre --------------
Extract: dow=[4] months=['09'] 16h–19h → 2000 pts
min_cluster_size=50  min_samples=5  method=leaf  → 11 clusters  |  1085 pts bruit (54.2%)


/home/gviel/app/miniconda3/envs/uber/lib/python3.12/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


---------- HC nuit jeudi septembre --------------
Extract: dow=[4] months=['09'] 19h–06h → 2000 pts
min_cluster_size=50  min_samples=5  method=leaf  → 9 clusters  |  814 pts bruit (40.7%)


/home/gviel/app/miniconda3/envs/uber/lib/python3.12/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


### 2.5 - Binning hexagonal avec H3

**H3** (Uber, 2018) est une bibliothèque de **découpage géographique hiérarchique** : la surface terrestre est divisée en hexagones imbriqués à 16 niveaux de résolution.

Contrairement à KMeans et DBSCAN, H3 ne *cherche* pas des groupes — il **compte** les pickups par case hexagonale puis colorie les cases selon leur densité.
C'est du **binning spatial** : simple, reproductible, sans paramètre de distance.

Paramètre principal : `resolution` (0 = continents, 15 = ~1 m²)
- résolution 7 ≈ 5 km² par hexagone (quartiers)
- résolution 8 ≈ 0,7 km² par hexagone (blocs de rues NYC)
- résolution 9 ≈ 0,1 km² par hexagone (intersections)

Avantages : pas d'hyperparamètre de distance, couverture exhaustive, hiérarchie naturelle.
Limite : ne détecte pas de formes irrégulières, et les zones peu denses sont toujours présentes (juste vides).


In [33]:
import h3
import plotly.graph_objects as go
import json

H3_RESOLUTION = 8      # 8 ≈ 0,7 km² / hexagone (blocs de rues NYC)
H3_SAMPLE_N   = None   # None = tous les points du sous-ensemble

for c in clusters_defs:
    print(f"---------- {c['name']} --------------")
    data_params = {k: v for k, v in c['params']['data'].items() if k != 'sample_n'}
    data_params['sample_n'] = H3_SAMPLE_N
    df_period = extract_period_df(df, **data_params)

    # Assigner chaque pickup à un hexagone H3
    df_period = df_period.copy()
    df_period['h3_cell'] = df_period.apply(
        lambda r: h3.latlng_to_cell(r['lat'], r['lon'], H3_RESOLUTION), axis=1
    )

    # Compter les pickups par hexagone
    hex_counts = df_period.groupby('h3_cell').size().reset_index(name='count')
    hex_counts = hex_counts.sort_values('count', ascending=False)
    print(f"  {len(hex_counts)} hexagones actifs  |  max={hex_counts['count'].max()}  médiane={hex_counts['count'].median():.0f}")

    # Construire les polygones GeoJSON des hexagones
    features = []
    for _, row in hex_counts.iterrows():
        boundary = h3.cell_to_boundary(row['h3_cell'])   # liste de (lat, lon)
        coords   = [[lon, lat] for lat, lon in boundary]
        coords.append(coords[0])                          # fermer le polygone
        features.append({
            "type": "Feature",
            "id": row['h3_cell'],
            "geometry": {"type": "Polygon", "coordinates": [coords]},
            "properties": {"count": int(row['count'])}
        })
    geojson = {"type": "FeatureCollection", "features": features}

    fig = go.Figure(go.Choroplethmapbox(
        geojson=geojson,
        locations=hex_counts['h3_cell'],
        z=hex_counts['count'],
        colorscale='YlOrRd',
        marker_opacity=0.7,
        marker_line_width=0,
        colorbar=dict(title='Pickups', thickness=12, len=0.5),
        hovertemplate='<b>%{location}</b><br>Pickups : %{z}<extra></extra>',
    ))
    fig.update_layout(
        mapbox=dict(style=MAP_STYLE, center=NY_CENTER, zoom=10),
        height=700, width=1000,
        title=f"H3 résolution {H3_RESOLUTION} — {c['name']}",
        margin=dict(l=0, r=0, t=40, b=0),
    )
    fig.show()
    fname = c['name'].replace(' ', '_')
    fig.write_html(f"exports/h3_{fname}.html")


---------- HPM matin jeudi septembre --------------
Extract: dow=[4] months=['09'] 06h–09h → 27369 pts
  774 hexagones actifs  |  max=789  médiane=2


/tmp/ipykernel_870172/4230798825.py:39: DeprecationWarning: *choroplethmapbox* is deprecated! Use *choroplethmap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = go.Figure(go.Choroplethmapbox(


---------- HC journée jeudi septembre --------------
Extract: dow=[4] months=['09'] 10h–15h → 39693 pts
  1020 hexagones actifs  |  max=2130  médiane=2


/tmp/ipykernel_870172/4230798825.py:39: DeprecationWarning: *choroplethmapbox* is deprecated! Use *choroplethmap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = go.Figure(go.Choroplethmapbox(


---------- HPS soir jeudi septembre --------------
Extract: dow=[4] months=['09'] 16h–19h → 44194 pts
  882 hexagones actifs  |  max=3045  médiane=2


/tmp/ipykernel_870172/4230798825.py:39: DeprecationWarning: *choroplethmapbox* is deprecated! Use *choroplethmap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = go.Figure(go.Choroplethmapbox(


---------- HC nuit jeudi septembre --------------
Extract: dow=[4] months=['09'] 19h–06h → 58564 pts
  1114 hexagones actifs  |  max=2374  médiane=2


/tmp/ipykernel_870172/4230798825.py:39: DeprecationWarning: *choroplethmapbox* is deprecated! Use *choroplethmap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = go.Figure(go.Choroplethmapbox(


# TODO
- template sur dow -> clusteriser
- reproduire sur chaque fichier mensuel
- mettre un slider
- regarder pour les heures...
- utiliser 2 méthodes KM + DBScan